
# Convert PicoQuant T2 <-> T3 record modes

Rebuild a ``TTTR`` in the other record mode.

tttrlib intentionally has no T2<->T3 conversion in C++: the transform is a small
NumPy operation on the bulk arrays, so it lives here as a reusable helper.

* **T2** events carry a single fine time tag (the macro time) and no micro time.
* **T3** events reference each photon to a laser sync period: the macro time is
  the (absolute) sync-period index and the micro time (``dtime``) is the delay
  within the period.

Both directions read the absolute ``macro_times``/``micro_times`` arrays from an
input ``TTTR`` and construct a *second* ``TTTR`` with the other representation.
The macro time of the T3 stream is an absolute sync index; the HHT3v2 writer
re-inserts the 10-bit sync-counter overflow records on write, so no explicit
overflow bookkeeping is needed here.

The HydraHarp/MultiHarp T3 ``dtime`` field is only 15 bits (0..32767). When the
sync period ``P`` exceeds that, ``dtime`` is *binned* (not clamped) so the whole
period stays representable; the micro-time resolution is scaled accordingly.


In [ ]:
import numpy as np
import tttrlib

# tttrlib record-type constants (see include/TTTRHeaderTypes.h)
PQ_RECORD_TYPE_HHT2v2 = 1
PQ_RECORD_TYPE_HHT3v2 = 4

# HydraHarp/MultiHarp T3 dtime field is 15-bit.
T3_DTIME_BITS = 15
T3_DTIME_CHANNELS = 1 << T3_DTIME_BITS  # 32768 (max dtime + 1)


def _arrays(tttr):
    """Pull the four bulk arrays with the dtypes the TTTR constructor expects."""
    macro = np.asarray(tttr.macro_times).astype(np.uint64)
    micro = np.asarray(tttr.micro_times).astype(np.uint64)
    routing = np.asarray(tttr.routing_channels).astype(np.int8)
    events = np.asarray(tttr.event_types).astype(np.int8)
    return macro, micro, routing, events


def _resolve_period(tttr, sync_rate, sync_period):
    """Sync period in macro-time (time-tag) units."""
    if sync_period is not None and sync_period > 0:
        return int(sync_period)
    if sync_rate is not None and sync_rate > 0:
        macro_res = tttr.header.macro_time_resolution
        if macro_res > 0:
            return int(round((1.0 / sync_rate) / macro_res))
    raise ValueError(
        "cannot determine the sync period: pass sync_period (macro-time units) "
        "or sync_rate (Hz) together with a macro_time_resolution in the header")


def t2_to_t3(tttr, sync_rate=None, sync_period=None):
    """Return a new T3 ``TTTR`` derived from a T2 ``TTTR``.

    ``n_sync = time_tag // P`` and ``dtime = (time_tag % P) // binning`` with
    ``binning = ceil(P / 32768)`` so ``dtime`` fits the 15-bit T3 field. Binning
    preserves the full sync period instead of clamping the tail of each period.
    """
    macro, _micro, routing, events = _arrays(tttr)
    P = _resolve_period(tttr, sync_rate, sync_period)
    if P <= 0:
        raise ValueError("sync period must be positive")

    n_sync = macro // np.uint64(P)
    dtime_full = macro % np.uint64(P)

    binning = 1
    if P > T3_DTIME_CHANNELS:
        binning = (P + T3_DTIME_CHANNELS - 1) // T3_DTIME_CHANNELS
    dtime = (dtime_full // np.uint64(binning)).astype(np.uint16)
    n_micro_channels = (P + binning - 1) // binning  # <= 32768

    out = tttrlib.TTTR(
        n_sync.astype(np.uint64), dtime,
        routing.astype(np.int8), events.astype(np.int8))

    macro_res = tttr.header.macro_time_resolution
    if macro_res <= 0:
        macro_res = 1.0
    out.header.tttr_record_type = PQ_RECORD_TYPE_HHT3v2
    out.header.set_micro_time_resolution(macro_res * binning)
    out.header.set_macro_time_resolution(macro_res * P)
    out.header.set_number_of_micro_time_channels(int(n_micro_channels))
    return out


def t3_to_t2(tttr, n_micro=None):
    """Return a new T2 ``TTTR`` derived from a T3 ``TTTR``.

    ``time_tag = n_sync * n_micro + dtime`` and ``micro_time = 0``. ``n_micro``
    defaults to the effective number of micro-time channels. The absolute
    arrival time is preserved at TAC resolution; the exact ``(n_sync, dtime)``
    split is only recoverable when every photon had ``dtime < n_micro``.
    """
    macro, micro, routing, events = _arrays(tttr)
    if n_micro is None:
        n_micro = int(tttr.header.get_effective_number_of_micro_time_channels())
    if n_micro <= 0:
        n_micro = 1

    time_tag = (macro * np.uint64(n_micro) + micro).astype(np.uint64)
    zero_micro = np.zeros(time_tag.shape, np.uint16)

    out = tttrlib.TTTR(
        time_tag, zero_micro, routing.astype(np.int8), events.astype(np.int8))
    micro_res = tttr.header.micro_time_resolution
    out.header.tttr_record_type = PQ_RECORD_TYPE_HHT2v2
    if micro_res > 0:
        out.header.set_macro_time_resolution(micro_res)
    out.header.set_number_of_micro_time_channels(1)
    return out


if __name__ == "__main__":
    import sys
    if len(sys.argv) < 3 or sys.argv[1] not in ("t2t3", "t3t2"):
        print("usage: t2_t3_conversion.py {t2t3|t3t2} IN.ptu OUT.ptu "
              "[sync_period]")
        raise SystemExit(2)
    mode, src, dst = sys.argv[1], sys.argv[2], sys.argv[3]
    data = tttrlib.TTTR(src)
    if mode == "t2t3":
        period = int(sys.argv[4]) if len(sys.argv) > 4 else None
        converted = t2_to_t3(data, sync_period=period)
    else:
        converted = t3_to_t2(data)
    converted.write(dst)
    print("wrote %s (%d events)" % (dst, converted.size()))